<a href="https://colab.research.google.com/github/5556mani/AI-Engineering/blob/main/PyTorch/7_ANN_training_using_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# we will be using the code from the previous file , and modify that to train in GPU

# we have to move model , data while training and eveluation to GPU using .to(device)

In [1]:
# import libraries
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as opt

In [2]:
device=('cuda' if torch.cuda.is_available() else 'cpu')
device

'cuda'

In [3]:
df=pd.read_csv("fmnist_small.csv")

In [4]:
# setting seed
torch.manual_seed(42)

In [5]:
# splitting the dataset in train and test
X=df.iloc[:,1:].values
Y=df.iloc[:,0].values
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

X_test=X_test/255.0
X_train=X_train/255.0



In [6]:
# writing custom class for dataset
class custom_DATA_class(Dataset):

  def __init__(self,features,labels):
    self.labels  =torch.tensor(labels,    dtype= torch.long)
    self.features=torch.tensor(features,  dtype= torch.float32  )

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    return self.features[index],self.labels[index]

In [7]:
# create train_dataset object
train_dataset=custom_DATA_class(X_train,Y_train)
# create test_dataset object
test_dataset = custom_DATA_class(X_test, Y_test)

In [8]:
# create Dataloader
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)

In [9]:
# define NN class
class NN(nn.Module):

  def __init__(self,inputshape):

    super().__init__()
    self.seq=nn.Sequential(
                          nn.Linear(inputshape,128),
                          nn.ReLU(),
                          nn.Linear(128,64),
                          nn.ReLU(),
                          nn.Linear(64,10)
    )


  def forward(self,features):
    return self.seq(features)

In [10]:
# hyperparameters
lr=0.01
epochs=10

In [11]:
# model , loss function , optimiser
model=NN(X_train.shape[1])
model=model.to(device)

loss_func=nn.CrossEntropyLoss()

optim=opt.SGD(model.parameters(), lr=lr)

In [12]:
# training loop
for epoch in range(epochs):
  epoch_loss=0
  for batch_features,batch_labels in train_loader:
    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

    #forward pass
    out=model(batch_features)

    # loss
    loss=loss_func(out,batch_labels)

    # back-propogation
    optim.zero_grad()
    loss.backward()

    #update
    optim.step()

    epoch_loss+=loss.item()
  avg_l=epoch_loss/len(train_loader)
  print(f"Epoch: {epoch+1}, Loss: {avg_l}")

Epoch: 1, Loss: 2.2626999939713524
Epoch: 2, Loss: 2.126983052102205
Epoch: 3, Loss: 1.8496980655973203
Epoch: 4, Loss: 1.507015058927447
Epoch: 5, Loss: 1.258176317838865
Epoch: 6, Loss: 1.0970075537111157
Epoch: 7, Loss: 0.9920613721152333
Epoch: 8, Loss: 0.9135949810810178
Epoch: 9, Loss: 0.8673653184810531
Epoch: 10, Loss: 0.8274898885566497


In [13]:
# eval mode
model.eval()

NN(
  (seq): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [14]:
# evaluation code
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:
    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)


0.6705882352941176
